# Arm 1: Primary Pool (FAA) Alone

**Objective.** Evaluate frontal alpha asymmetry (FAA) as a standalone predictor of rTMS responder status, using four classifiers in parallel (Ledoit-Wolf shrinkage LDA, elastic-net logistic regression, Bayesian logistic regression, unregularized logistic regression), with nested cross-validation and permutation testing for each. All four results are reported. This is Arm 1 of six pre-specified arms (Decision 5, `modelling_decisions.md`), reported in full regardless of outcome.

This notebook also builds and validates the fold-scoped age-regression transformer and the generalized nested-CV harness, using this arm's single-feature case as the simplest test before refactoring both into `src/modelling.py` for reuse across Arms 2–6.

**Inputs.**
- `full_cohort_features.parquet`, restEC/heog_off, subset to alpha power (8–13 Hz) at F3 and F4
- Responder/non-responder labels
- Age, for fold-scoped deconfounding

**Feature construction.** FAA = alpha power(F4) − alpha power(F3), raw difference (not log-transformed - explicit exception to Decision 2's default, since FAA is a signed difference and log is undefined for negative values), matching Provaznikova et al. (2025) exactly (Decision 5).

**Assumptions.**
- Age is regressed out of FAA within training folds only (out-of-sample deconfounding, Chyzhyk et al., 2022), never fit on train and test jointly. Age is significantly associated with responder status in this cohort (t = -2.68, p = 0.01), a real, not hypothetical, risk that this step could remove some outcome-related signal along with the confound (Decision 7).
- Outer CV loop and transformer calls are explicit, not routed through sklearn `Pipeline`, since age needs to reach the transformer as a plain argument rather than a smuggled feature column.
- Bailey et al.'s theta connectivity/alpha power construct is deliberately excluded from this arm, specified, but already non-replicated at scale, so it's a separate supplementary test rather than part of the primary pool (Decision 5).
- Both accuracy and AUC are reported for every classifier: accuracy is primary (matches Chang et al., 2025), AUC is secondary (comparability with Provaznikova's FAA evidence).

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, balanced_accuracy_score

## Data loading and feature construction

In [2]:
# Temporary bootstrap path, just to make src/ importable — not the real project root
sys.path.insert(0, str(Path.cwd().parent))

from src.preprocessing import find_repo_root
project_root = find_repo_root()
#Define dir 
data_dir = project_root / "data"
features_dir = data_dir / "features"

full_df = pd.read_parquet(features_dir / "full_cohort_features.parquet")
alpha_power_columns = full_df[['F3_alpha_power', 'F4_alpha_power']]
print(alpha_power_columns)

     F3_alpha_power  F4_alpha_power
0        135.931109      141.647315
1         47.927017       52.177002
2         41.122609       40.342005
3        281.296923      308.598717
4        181.383911      170.625480
..              ...             ...
155      283.047167      262.343693
156       38.087251       38.843060
157      294.013122      321.013081
158      291.411640      323.286509
159       99.080413       96.545709

[160 rows x 2 columns]


In [3]:
# Define an explicit QC & feature-columns lists 
id_qc_cols = [
    'subject_id', 'condition', 'variant', 'reason', 'heog_variant',
    'preprocessing_status', 'n_epochs_before', 'n_epochs_after',
    'output_path', 'autoreject_consensus', 'autoreject_n_interpolate',
    'autoreject_extreme', 'heog_n_candidates', 'heog_n_valid',
    'heog_correction_applied', 'preprocessing_error',
]

feature_cols = [c for c in full_df.columns if c not in id_qc_cols]

#valudate
print(len(id_qc_cols))                                   # expect 16
print(set(id_qc_cols) - set(full_df.columns))             # expect empty - anything here is a typo
print(len(feature_cols))                                  # expect 5015

16
set()
5015


In [4]:
#extracting age and responder statues
cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")
print(cohort_df.columns.tolist())
print(len(cohort_df))

# does it actually give one row per subject, or does it have the same duplication risk?
print(cohort_df.iloc[:, 0].duplicated().sum()) 

['TDBRAIN_ID', 'DISC/REP', 'indication', 'formal_status', 'Dataset', 'Consent', 'sessSeason', 'sessTime', 'Responder', 'Remitter', 'age', 'gender', 'sessID', 'nrSessions', 'neoFFI_q1', 'neoFFI_q2', 'neoFFI_q3', 'neoFFI_q4', 'neoFFI_q5', 'neoFFI_q6', 'neoFFI_q7', 'neoFFI_q8', 'neoFFI_q9', 'neoFFI_q10', 'neoFFI_q11', 'neoFFI_q12', 'neoFFI_q13', 'neoFFI_q14', 'neoFFI_q15', 'neoFFI_q16', 'neoFFI_q17', 'neoFFI_q18', 'neoFFI_q19', 'neoFFI_q20', 'neoFFI_q21', 'neoFFI_q22', 'neoFFI_q23', 'neoFFI_q24', 'neoFFI_q25', 'neoFFI_q26', 'neoFFI_q27', 'neoFFI_q28', 'neoFFI_q29', 'neoFFI_q30', 'neoFFI_q31', 'neoFFI_q32', 'neoFFI_q33', 'neoFFI_q34', 'neoFFI_q35', 'neoFFI_q36', 'neoFFI_q37', 'neoFFI_q38', 'neoFFI_q39', 'neoFFI_q40', 'neoFFI_q41', 'neoFFI_q42', 'neoFFI_q43', 'neoFFI_q44', 'neoFFI_q45', 'neoFFI_q46', 'neoFFI_q47', 'neoFFI_q48', 'neoFFI_q49', 'neoFFI_q50', 'neoFFI_q51', 'neoFFI_q52', 'neoFFI_q53', 'neoFFI_q54', 'neoFFI_q55', 'neoFFI_q56', 'neoFFI_q57', 'neoFFI_q58', 'neoFFI_q59', 'neoFFI_q60

In [5]:
# Build FAA
faa = alpha_power_columns['F4_alpha_power'] - alpha_power_columns['F3_alpha_power']

In [6]:
# Merge FAA with age and responder status, keyed on subject ID
model_df = pd.DataFrame({
    'subject_id': full_df['subject_id'],
    'FAA': faa,
})

model_df = model_df.merge(
    cohort_df[['TDBRAIN_ID', 'age', 'Responder']],
    left_on='subject_id', right_on='TDBRAIN_ID', how='left'
).drop(columns='TDBRAIN_ID')

print(len(model_df))                                       # must stay 160
print(model_df[['FAA', 'age', 'Responder']].isna().sum())  # must all be 0
print(model_df['Responder'].value_counts())                # sanity check
model_df.head()

160
FAA          0
age          0
Responder    0
dtype: int64
Responder
1    93
0    67
Name: count, dtype: int64


,subject_id,FAA,age,Responder
0,sub-88045809,5.716206,43.06,0
1,sub-88022765,4.249985,32.96,1
2,sub-88023485,-0.780604,44.50,1
3,sub-88061061,27.301794,27.69,1
4,sub-88021321,-10.758431,28.28,0


In [7]:
# Import AgeDeconfounder from src/modelling.py and validate
from src.modelling import AgeDeconfounder

# Whole-cohort check against scipy.stats.linregress
deconf = AgeDeconfounder()
deconf.fit(model_df[['FAA']].values, model_df['age'].values)

ref = stats.linregress(model_df['age'], model_df['FAA'])

print("slope diff:", deconf.slopes_[0] - ref.slope)
print("intercept diff:", deconf.intercepts_[0] - ref.intercept)

residuals = deconf.transform(model_df[['FAA']].values, model_df['age'].values)
print("residual sum:", residuals.sum())

# Fold-scoped check: train-fitted coefficients applied to test should NOT
# zero out, and should differ from test's own independently-fitted slope
half = len(model_df) // 2
train_half = model_df.iloc[:half]
test_half = model_df.iloc[half:]

deconf_fold = AgeDeconfounder()
deconf_fold.fit(train_half[['FAA']].values, train_half['age'].values)

train_resid = deconf_fold.transform(train_half[['FAA']].values, train_half['age'].values)
test_resid  = deconf_fold.transform(test_half[['FAA']].values, test_half['age'].values)

independent_test_fit = AgeDeconfounder()
independent_test_fit.fit(test_half[['FAA']].values, test_half['age'].values)

print("train residual sum:", train_resid.sum())
print("test residual sum:", test_resid.sum())
print("train-fitted slope applied to test:", deconf_fold.slopes_[0])
print("test's own independently-fitted slope:", independent_test_fit.slopes_[0])

slope diff: 1.3877787807814457e-17
intercept diff: -8.881784197001252e-16
residual sum: 7.105427357601002e-14
train residual sum: -4.263256414560601e-14
test residual sum: -224.7870045912082
train-fitted slope applied to test: -0.19455396514821635
test's own independently-fitted slope: 0.07290524760563255


## Nested CV for the 4 classifiers

In [8]:
# Nested CV: core loop (accuracy + AUC per classifier per fold, no permutation testing yet)

from src.modelling import run_nested_cv

N_OUTER_SPLITS = 5   # n=160; keeps test folds ~32 subjects, workable given class balance
N_INNER_SPLITS = 3   # tuning loop, within each outer training fold only
RANDOM_STATE = 42

X = model_df[['FAA']].values
y = model_df['Responder'].values
age = model_df['age'].values

# name -> (estimator, param_grid or None if it needs no tuning)
classifier_specs = {
    'LDA (Ledoit-Wolf)': (
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
        None
    ),
    'Logistic (unregularized)': (
        LogisticRegression(C=np.inf, max_iter=1000),
        None
    ),
    'Logistic (elastic-net)': (
        LogisticRegression(solver='saga', max_iter=5000, random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.5, 0.9]}
    ),
    'Logistic (L2 / "Bayesian" MAP)': (
        LogisticRegression(l1_ratio=0, max_iter=1000),
        {'C': [0.01, 0.1, 1, 10, 100]}
    ),
}

results_df = run_nested_cv(X, y, age, classifier_specs, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)

print(results_df.groupby('classifier')[['balanced_accuracy', 'accuracy', 'auc', 'sensitivity', 'specificity', 'ppv']].mean())
results_df

                                balanced_accuracy  accuracy       auc  \
classifier                                                              
LDA (Ledoit-Wolf)                        0.510947   0.56875  0.638873   
Logistic (L2 / "Bayesian" MAP)           0.518640   0.57500  0.638873   
Logistic (elastic-net)                   0.552522   0.60000  0.638873   
Logistic (unregularized)                 0.513376   0.56875  0.638873   

                                sensitivity  specificity       ppv  
classifier                                                          
LDA (Ledoit-Wolf)                  0.871345     0.150549  0.588704  
Logistic (L2 / "Bayesian" MAP)     0.871345     0.165934  0.592842  
Logistic (elastic-net)             0.838012     0.267033  0.614861  
Logistic (unregularized)           0.860819     0.165934  0.590132  


,classifier,fold,balanced_accuracy,accuracy,auc,sensitivity,specificity,ppv
0,LDA (Ledoit-Wolf),0,0.444444,0.50000,0.539683,0.888889,0.000000,0.533333
1,Logistic (unregularized),0,0.444444,0.50000,0.539683,0.888889,0.000000,0.533333
2,Logistic (elastic-net),0,0.539683,0.56250,0.539683,0.722222,0.357143,0.590909
3,"Logistic (L2 / ""Bayesian"" MAP)",0,0.444444,0.50000,0.539683,0.888889,0.000000,0.533333
4,LDA (Ledoit-Wolf),1,0.551587,0.59375,0.658730,0.888889,0.214286,0.592593
5,Logistic (unregularized),1,0.551587,0.59375,0.658730,0.888889,0.214286,0.592593
6,Logistic (elastic-net),1,0.587302,0.62500,0.658730,0.888889,0.285714,0.615385
7,"Logistic (L2 / ""Bayesian"" MAP)",1,0.551587,0.59375,0.658730,0.888889,0.214286,0.592593
8,LDA (Ledoit-Wolf),2,0.497976,0.56250,0.639676,0.842105,0.153846,0.592593
9,Logistic (unregularized),2,0.497976,0.56250,0.639676,0.842105,0.153846,0.592593


### Class-balanced weighting

The unweighted run above shows a clear majority-class bias: sensitivity 0.77–0.87 against specificity only 0.15–0.34 across all four classifiers, and balanced accuracy (0.51–0.56) sitting close to chance despite raw accuracy (0.57–0.59) looking more favorable, a known artifact of evaluating an imbalance-blind objective (this cohort is 93 responders / 67 non-responders) with a metric that doesn't correct for it.

Rather than tune this after seeing whether it helps the permutation test clear significance, which would reopen the same post-hoc-selection risk the six-arm pre-specification and the bounded literature search were built to avoid, class-balanced weighting is adopted here as a standing decision, on the diagnostic evidence above, before any significance testing is run. `class_weight='balanced'` is applied to all three logistic variants; `LinearDiscriminantAnalysis` has no `class_weight` parameter, so `priors=[0.5, 0.5]` is used instead, overriding sklearn's default of estimating priors from the training fold's empirical class frequencies, the same majority-class-driven behavior this change is meant to correct.

This changes what each classifier optimizes for, not just how its output is scored, so results below are not directly comparable to the unweighted run above; both are kept in the notebook rather than overwriting, so the reasoning for the change stays visible.

In [9]:
# Nested CV, re-run with class-balanced weighting

classifier_specs_balanced = {
    'LDA (Ledoit-Wolf, balanced priors)': (
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto', priors=[0.5, 0.5]),
        None
    ),
    'Logistic (unregularized, balanced)': (
        LogisticRegression(C=np.inf, max_iter=1000, class_weight='balanced'),
        None
    ),
    'Logistic (elastic-net, balanced)': (
        LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.5, 0.9]}
    ),
    'Logistic (L2 / "Bayesian" MAP, balanced)': (
        LogisticRegression(l1_ratio=0, max_iter=1000, class_weight='balanced'),
        {'C': [0.01, 0.1, 1, 10, 100]}
    ),
}

results_balanced_df = run_nested_cv(X, y, age, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)

print(results_balanced_df.isna().sum())   # check for degenerate folds before trusting the means
print(results_balanced_df.groupby('classifier')[['balanced_accuracy', 'accuracy', 'auc', 'sensitivity', 'specificity', 'ppv']].mean())
results_balanced_df

classifier           0
fold                 0
balanced_accuracy    0
accuracy             0
auc                  0
sensitivity          0
specificity          0
ppv                  0
dtype: int64
                                          balanced_accuracy  accuracy  \
classifier                                                              
LDA (Ledoit-Wolf, balanced priors)                 0.570609   0.58125   
Logistic (L2 / "Bayesian" MAP, balanced)           0.570609   0.58125   
Logistic (elastic-net, balanced)                   0.587032   0.59375   
Logistic (unregularized, balanced)                 0.570609   0.58125   

                                               auc  sensitivity  specificity  \
classifier                                                                     
LDA (Ledoit-Wolf, balanced priors)        0.638873     0.645614     0.495604   
Logistic (L2 / "Bayesian" MAP, balanced)  0.638873     0.645614     0.495604   
Logistic (elastic-net, balanced)          0.

,classifier,fold,balanced_accuracy,accuracy,auc,sensitivity,specificity,ppv
0,"LDA (Ledoit-Wolf, balanced priors)",0,0.519841,0.53125,0.539683,0.611111,0.428571,0.578947
1,"Logistic (unregularized, balanced)",0,0.519841,0.53125,0.539683,0.611111,0.428571,0.578947
2,"Logistic (elastic-net, balanced)",0,0.527778,0.53125,0.539683,0.555556,0.500000,0.588235
3,"Logistic (L2 / ""Bayesian"" MAP, balanced)",0,0.519841,0.53125,0.539683,0.611111,0.428571,0.578947
4,"LDA (Ledoit-Wolf, balanced priors)",1,0.539683,0.56250,0.658730,0.722222,0.357143,0.590909
5,"Logistic (unregularized, balanced)",1,0.539683,0.56250,0.658730,0.722222,0.357143,0.590909
6,"Logistic (elastic-net, balanced)",1,0.575397,0.59375,0.658730,0.722222,0.428571,0.619048
7,"Logistic (L2 / ""Bayesian"" MAP, balanced)",1,0.539683,0.56250,0.658730,0.722222,0.357143,0.590909
8,"LDA (Ledoit-Wolf, balanced priors)",2,0.597166,0.59375,0.639676,0.578947,0.615385,0.687500
9,"Logistic (unregularized, balanced)",2,0.597166,0.59375,0.639676,0.578947,0.615385,0.687500


In [10]:
# Diagnostic: confirm LDA/unregularized/L2's tie is real convergence, not a bug
outer_cv_check = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_idx_to_check = 1  # elastic-net differed here - informative fold to inspect
train_idx, test_idx = list(outer_cv_check.split(X, y))[fold_idx_to_check]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
age_train, age_test = age[train_idx], age[test_idx]

deconf = AgeDeconfounder()
deconf.fit(X_train, age_train)
X_train_clean = deconf.transform(X_train, age_train)
X_test_clean = deconf.transform(X_test, age_test)

for clf_name, (estimator, param_grid) in classifier_specs_balanced.items():
    if param_grid is not None:
        inner_cv = StratifiedKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(estimator, param_grid, cv=inner_cv, scoring='balanced_accuracy')
        search.fit(X_train_clean, y_train)
        fitted_model = search.best_estimator_
        print(clf_name, "best params:", search.best_params_)
    else:
        fitted_model = estimator.fit(X_train_clean, y_train)

    proba = fitted_model.predict_proba(X_test_clean)[:, 1]
    print(clf_name)
    print("  probabilities:", np.round(proba, 3))
    print()

LDA (Ledoit-Wolf, balanced priors)
  probabilities: [0.575 0.483 0.532 0.532 0.522 0.521 0.395 0.669 0.391 0.552 0.469 0.535
 0.524 0.514 0.504 0.377 0.529 0.504 0.425 0.482 0.529 0.498 0.507 0.399
 0.527 0.505 0.366 0.513 0.504 0.501 0.578 0.669]

Logistic (unregularized, balanced)
  probabilities: [0.579 0.482 0.534 0.534 0.523 0.522 0.39  0.677 0.386 0.555 0.467 0.537
 0.525 0.515 0.504 0.371 0.53  0.504 0.422 0.481 0.531 0.498 0.507 0.394
 0.529 0.505 0.359 0.513 0.504 0.501 0.582 0.676]

Logistic (elastic-net, balanced) best params: {'C': 0.01, 'l1_ratio': 0.1}
Logistic (elastic-net, balanced)
  probabilities: [0.569 0.481 0.528 0.528 0.518 0.517 0.398 0.659 0.394 0.547 0.468 0.531
 0.52  0.511 0.501 0.38  0.525 0.502 0.427 0.48  0.526 0.496 0.504 0.402
 0.524 0.502 0.37  0.51  0.502 0.498 0.572 0.659]

Logistic (L2 / "Bayesian" MAP, balanced) best params: {'C': 0.01}
Logistic (L2 / "Bayesian" MAP, balanced)
  probabilities: [0.577 0.482 0.533 0.533 0.522 0.522 0.392 0.674 0.388 0

### Findings so far, and next step

The near-identical probabilities across LDA, unregularized, and L2 logistic (visible above) are genuine convergence, not a bug - confirmed by checking the raw values rather than just the classifications: four independently-fit models landing within ~1 percentage point of each other, likely pulled together by both tuned models' grid search selecting maximal shrinkage (`C=0.01`). Class-balanced weighting fixed the majority-class bias seen in the unweighted run (specificity moved from ~0.15–0.34 up to ~0.36–0.69).
None of this, including elastic-net's standout 0.587 balanced accuracy, has been tested against chance yet. Next: permutation testing, shuffling responder labels many times and rerunning the full nested-CV loop each time, to see whether the observed balanced accuracy for each classifier is actually distinguishable from what random labels would produce at this sample size.

In [11]:
# Permutation test: is each classifier's mean balanced accuracy distinguishable from chance?

N_PERMUTATIONS = 1000
rng = np.random.RandomState(RANDOM_STATE)

# Observed statistic: reuse what's already computed, don't rerun
observed = results_balanced_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()

# Null distribution: same procedure, shuffled labels
print(f"N_PERMUTATIONS = {N_PERMUTATIONS}")

null_scores = {name: [] for name in classifier_specs_balanced}

for i in range(N_PERMUTATIONS):
    y_shuffled = rng.permutation(y)
    perm_df = run_nested_cv(X, y_shuffled, age, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_result = perm_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()
    for name, score in perm_result.items():
        null_scores[name].append(score)

# p-value: fraction of the null distribution at or above the observed value
# (+1/+1 correction: the observed result is itself one valid draw under the null)
print(f"{'classifier':<45} {'observed':>10} {'null mean':>10} {'p-value':>10}")
for name in classifier_specs_balanced:
    null_arr = np.array(null_scores[name])
    p_value = (np.sum(null_arr >= observed[name]) + 1) / (N_PERMUTATIONS + 1)
    print(f"{name:<45} {observed[name]:>10.4f} {null_arr.mean():>10.4f} {p_value:>10.4f}")

N_PERMUTATIONS = 1000
classifier                                      observed  null mean    p-value
LDA (Ledoit-Wolf, balanced priors)                0.5706     0.4974     0.0500
Logistic (unregularized, balanced)                0.5706     0.4976     0.0519
Logistic (elastic-net, balanced)                  0.5870     0.4989     0.0160
Logistic (L2 / "Bayesian" MAP, balanced)          0.5706     0.4976     0.0519


### Interpretation

Elastic-net logistic regression is the only one of the four classifiers to clear conventional significance (balanced accuracy 0.587, p = 0.016). The other three; LDA, unregularized logistic, and L2 logistic, sit right at the boundary (0.0500–0.0519), effectively indistinguishable from chance at a 0.05 threshold. Per Decision 5, no classifier is selected as a winner: all four are reported together, and the honest summary is *weak, borderline evidence* that FAA carries some signal for responder status in this cohort, not a clean positive result.

The effect size is modest even where significant - elastic-net's 0.587 is roughly 9 points above the 0.50 chance level, well below Provaznikova et al.'s (2025) reported AUC of 0.75-0.81. This notebook's own AUC (0.639, averaged across folds) is closer to that comparison, but sits meaningfully below it, and shouldn't be read as replicating Provaznikova's finding at similar strength.

Two structural findings from this arm, both genuine, not artifacts:
- **AUC is identical across all four classifiers, in every fold.** With one input feature, any monotonic classifier produces the same rank-ordering of subjects, which forces identical ROC curves. This is specific to p=1 and isn't expected to hold once Arm 2 adds Kuramoto features.
- **Class-balanced weighting changed the result.** The unweighted run showed high sensitivity (0.77-0.87) paired with low specificity (0.15-0.34), a majority-class bias, not real discriminative performance, confirmed by balanced accuracy sitting near chance (0.51-0.52) despite raw accuracy looking more favorable (0.57-0.59). Balancing was adopted as a standing decision before permutation testing, precisely to avoid tuning toward whichever configuration cleared significance (Decision 5).

The permutation procedure itself checks out: null-distribution means sit at 0.497-0.499 across all four classifiers, consistent with the ~0.50 expected under shuffled labels. 

## Summary

**Findings.** FAA shows weak evidence of association with rTMS responder status in this cohort: elastic-net logistic regression clears conventional significance (balanced accuracy 0.587, p = 0.016); LDA, unregularized, and L2 logistic sit at the 0.05 boundary and don't. No classifier is treated as a winner, per Decision 5 - all four are reported. Effect size is modest and below Provaznikova et al.'s (2025) reported strength. Class-balanced weighting (`class_weight='balanced'`; `priors=[0.5, 0.5]` for LDA) was necessary and is now a standing decision across all arms, documented in `modelling_decisions.md`. AUC is identical across all four classifiers in every fold, an expected consequence of p=1, not a bug, and not expected to recur once Arm 2 introduces additional features.

**Residual assumptions, carried into Arm 2 onward.**
- `AgeDeconfounder` and the nested-CV pattern are validated here and moved to `src/modelling.py`; Arm 2 imports rather than redefines.
- The explicit outer-loop design (transformer called directly, not through `Pipeline`) carries forward, since age still needs to reach the transformer as a plain argument.
- Same `StratifiedKFold` splits are reused across all four classifiers within a fold structure, so any between-classifier difference reflects the classifier, not the split.
- Hyperparameter grids for elastic-net and L2 (`C`, `l1_ratio`) are reasoned defaults, not independently validated - worth revisiting if results in later arms look sensitive to grid boundaries.
- Reproducibility was checked only at the fixed seed (`RANDOM_STATE = 42`); sensitivity to seed choice itself hasn't been tested and is an open question, not resolved here.
- The Bailey construct remains excluded from the primary pool and is deferred to its own supplementary test (Decision 5), not part of any arm's classifier comparison.